# Multi-Tier Backend Selection & Verification

This notebook verifies the automatic backend failover logic in `pk_pipeline/ml_services.py`:
1. **Priority 1 (Local):** `http://127.0.0.1:30000` (Local SGLang instance)
2. **Priority 2 (Remote DGX):** `http://100.85.128.70:30001` (Direct network / Tailscale VPN access)
3. **Priority 3 (API Mode):** OpenAI-compatible endpoint configured in `.env`
4. **Priority 4 (Embedded):** In-process `sgl.Engine` fallback

### Step 1: Inspect Loaded Environment Variables from `.env`

In [ ]:
from pk_pipeline.config import Config

print("Loaded APEX v2 Configuration:")
print(f"  • Local SGLang URL:  {Config.SGLANG_LOCAL_URL}")
print(f"  • DGX SGLang URL:    {Config.DGX_SGLANG_URL}")
print(f"  • OpenAI Base URL:   {Config.OPENAI_BASE_URL}")
print(f"  • OpenAI API Key:    {Config.OPENAI_API_KEY[:4]}... (length: {len(Config.OPENAI_API_KEY)})")
print(f"  • Target Model:      {Config.OPENAI_MODEL}")
print(f"  • Force Backend:     {Config.FORCE_BACKEND}")

### Step 2: Initialize `SGLangExtractor` with Automatic Backend Detection

In [ ]:
from pk_pipeline.ml_services import SGLangExtractor

extractor = SGLangExtractor()

print(f"\nExtractor Loaded:  {extractor.is_loaded}")
print(f"Selected Backend:  {extractor.backend_mode}")
print(f"Active Server URL: {extractor.server_url}")
print(f"Active Model:      {extractor.model_name}")

### Step 3: Run a Structured Extraction Test via Selected Backend

In [ ]:
import asyncio

sample_table = """
Table 1. Pharmacokinetic parameters for PFOS in human adult.
Body weight (BW): 70 kg
Cardiac output (QCC): 16.5 L/h/kg^0.74
Serum elimination clearance (CL): 0.0001 L/h/kg
Volume of distribution (Vd): 0.23 L/kg
"""

result = await extractor.extract_from_text(
    sample_table,
    role="table_xml",
    chunk_title="Test Human Adult PFOS Table"
)

print(f"Classification: {result.model_classification.model_type}")
print(f"Blocks Extracted: {len(result.parameter_blocks)}")

for block in result.parameter_blocks:
    print(f"\nBlock: {block.block_name} (Species: {block.implied_species})")
    for p in block.parameters:
        val = p.quantitative_data.value if hasattr(p.quantitative_data, 'value') else p.quantitative_data.get('value')
        unit = p.quantitative_data.unit if hasattr(p.quantitative_data, 'unit') else p.quantitative_data.get('unit')
        print(f"  • {p.parameter_name_standardized or p.parameter_name}: {val} {unit}")